## Packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

### Data Collection

In [20]:
data = pd.read_csv('/home/bibekg/Learning/AQI_COL/Air-Quality-Prediction/airprediction/Data/Real-Data/Real_Combine.csv')

# Reading first 5 rows
data.head()

,T,TM,Tm,SLP,H,VV,V,VM,AQI
0,7.4,9.8,4.8,1017.6,93.0,0.5,4.3,9.4,219.720833
1,7.8,12.7,4.4,1018.5,87.0,0.6,4.4,11.1,182.187500
2,6.7,13.4,2.4,1019.4,82.0,0.6,4.8,11.1,154.037500
3,8.6,15.5,3.3,1018.7,72.0,0.8,8.1,20.6,223.208333
4,12.4,20.9,4.4,1017.3,61.0,1.3,8.7,22.2,200.645833


In [ ]:
data.isnull().sum()

# if we have minimum null values we can drop null values
data = data.dropna()
# We don`t have null values

# Plotting the Null values
sns.heatmap(data.isnull(), yticklabels=False, cbar=False, cmap='viridis')

### Dividing the features

In [ ]:
X = data.drop(['PM 2.5'], axis=1) # Droping the Target Variable i.e. PM 2.5
Y = pd.DataFrame(data['PM 2.5']) # Assigning the Target Vatiavle to Y

### Feature Importance
You can get the feature importance of each feature of your dataset by using the feature importance property of the model.

Feature importance gives you a score for each feature of your data, the higher the score more important or relevant is the feature towards your output variable.

Feature importance is an inbuilt class that comes with Tree Based Regressor, we will be using Extra Tree Regressor for extracting the top 10 features for the dataset.

In [ ]:
from sklearn.ensemble  import ExtraTreesRegressor

In [ ]:
model = ExtraTreesRegressor()
model.fit(X,Y)

In [ ]:
# This is used when we have many features and we have to select top 10 featues from the DataSet
fea_impotance = pd.DataFrame({
                                 'Features': X.columns,
                                 'Feature_Imp' : model.feature_importances_
                                 })
fea_impotance

In [ ]:
# Plotting the Graph of FEATURE IMPORTANCE for better visualisation
feat_imp = pd.Series(model.feature_importances_, index=X.columns)
feat_imp.nlargest(5).plot(kind='barh')
plt.show()

### Train Test Data Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train,Y_test = train_test_split(X, Y, test_size=0.3, train_size=0.7, random_state=0)

### Plotting Target Variable

In [ ]:
sns.distplot(Y)

## Implementing Desicion Tree Regression

In [ ]:
from sklearn.tree import DecisionTreeRegressor

In [ ]:
dtree = DecisionTreeRegressor(criterion='mse')

dtree.fit(X_train, Y_train)

In [ ]:
# Calcu;ating R^2 for Training dataset
dtree_train_score = dtree.score(X_train, Y_train)
print(f'Coefficient of determination R^2 on training Dataset : {dtree_train_score}')

In [ ]:
# Calcu;ating R^2 for Testing dataset
dtree_test_score = dtree.score(X_test, Y_test)
print(f'Coefficient of determination R^2 on training Dataset : {dtree_test_score}')

From this we can the R^2 value for training dataset is 100%

and R^2 for testing dataset is 24%

i.e. this is a overfitting problem

In [ ]:
# Performing cross validation
from sklearn.model_selection import cross_val_score
dtree_cross_val_score = cross_val_score(dtree, X,Y, cv=5)

In [ ]:
dtree_cross_val_score.mean()

## Decision Tree Visualisation

In [ ]:
from IPython.display import Image
from six import StringIO

from sklearn.tree import export_graphviz
import pydotplus

In [ ]:
features = list(data.columns[:-1])
features

In [ ]:
# StringIO is generally used for Display Console
dot_data = StringIO()
export_graphviz(dtree, out_file=dot_data, feature_names= features, filled=True, rounded=True)

graph = pydotplus.graph_from_dot_data(dot_data.getvalue())
Image(graph.create_png())

## Model Evaluation

In [ ]:
prediction = dtree.predict(X_test)

In [ ]:
plt.scatter(Y_test,prediction)

### HyperParameter Tuning Decision Tree

In [ ]:
# Hyper Parameter Optimization
DecisionTreeRegressor()

# All these parameters are the Parameters of Decision Tree Regressor
params = {
    'splitter': ['best', 'random'], 
    'max_depth': [3,4,5,6,7,8,9,10,11,12,13,14,15,16], 
    'min_samples_leaf' : [1,2,3,4,5],
    'min_weight_fraction_leaf' : [0.1,0.2,0.3,0.4],
    'max_features' : ['auto', 'log2','sqrt',None],
    'max_leaf_nodes': [None,10,20,30,40,50,60,70]
}

In [ ]:
# For selecting the best Parameters
from sklearn.model_selection import GridSearchCV

In [ ]:
random_search = GridSearchCV(dtree, param_grid= params, scoring='neg_mean_squared_error', n_jobs=-1, cv=10, verbose=3)

In [ ]:
### creating a timer
def timer(start_time=None):
    if not start_time:
        start_time = datetime.now()
    elif start_time:
        thour, temp_sec = divmod((datetime.now() - start_time).total_seconds(),3600)
        tmin, tsec = divmod(temp_sec,60)
        print(f'\nTime Taken: {thour}hours {tmin}minutes {round(tsec,2)} seconds')

In [ ]:
from datetime import datetime

start_time = timer(None)
random_search.fit(X,Y)
timer(start_time)

In [ ]:
random_search.best_params_

In [ ]:
random_search.best_score_

In [ ]:
prediction = random_search.predict(X_test)

In [ ]:
plt.scatter(Y_test,prediction)

In [ ]:
# Calculting some of the Errors
from sklearn import metrics

In [ ]:
print(f'MAE: {metrics.mean_absolute_error(Y_test,prediction)}')
print(f'MSE: {metrics.mean_squared_error(Y_test,prediction)}')
print(f'RMSE: {np.sqrt(metrics.mean_squared_error(Y_test,prediction))}')

### Creating PKL FileFormat for deployment purpose

In [ ]:
import pickle

In [ ]:
# Open a file 
file = open('regression_decision_tree.pkl', 'wb')

# dump information to that file
pickle.dump(random_search,file)